#  Pure data case

We have three hyperparameters for the model:

- Relationship between $\sigma_s$ and $\sigma_0$
  - These follow the relation $\sigma_x^2 = \sigma_0^2\sigma_s^2$
  - We can set one of them to $c\sigma_{\epsilon}$, where $c$ is some constant deciding how you want to tune reconstruction penalty v/s regularisation
  - Or we can make them equal, and set both as: $\sigma_0 = \sigma_s = \sigma_x$
  - This removes $\sigma_{\epsilon}$ from the equation, do note that $\sigma_{\epsilon}$ is itself set with respect to $\sigma_x$, which we assume has 1 std.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import shelve
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import *
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
)
from pt_to_api import disjoint_ae, disjoint_ae_learned_sig
from sklearn.metrics.pairwise import (
    pairwise_distances,
    cosine_similarity,
    cosine_distances,
)
from scipy.optimize import linear_sum_assignment
from collections import defaultdict
import numpy as np
from torch import nn
from torch import optim
import warnings
from dataclasses import dataclass
from typing import Any
import math
import gc
import pandas as pd
from pt_to_api import benchmark as B
import ast

MODE = "light"
SHELVE_CACHE_ROOT = Path.cwd() / "pure-sparse-v3"
SHELVE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

# Start experiment

In [ ]:
def get_device(dim):
    if dim < 100:
        return "cpu"
    else:
        return "mps"



def get_metrics_df(metrics):
    mets = []
    for k, v in metrics.items():
        k = ast.literal_eval(k)
        m = v.copy()
        m["dims"] = k[0]
        m["atom_ratio"] = k[1]
        m["n_sample_ratio"] = k[2]
        m["sparsity_ratio"] = k[3]
        m["term3"] = k[4]
        mets.append(m)
    
    df = pd.DataFrame(mets)
    df = df.drop(columns=["vec_sim"])
    
    return df

In [ ]:
def make_dim_partition(patch_dim, n_components, seed=42):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(patch_dim)
    return [list(perm[i::n_components]) for i in range(n_components)]

def generate_synthetic_patches_sparse_atoms(
    patch_dim=72,
    n_components=10,
    k=3,
    k_atom=3,
    n_samples=1000,
    noise_std=0.01,
    seed=42,
    sigma_x=1,
):
    print(f"data generation: patch_dim={patch_dim} n_comps={n_components} k={k} k_atom={k_atom} n_samples={n_samples}")
    rng = np.random.RandomState(seed)
    dim_partition = make_dim_partition(patch_dim, n_components, seed)

    W_true = np.zeros((n_components, patch_dim))
    for i, dims in enumerate(dim_partition):
        n_active = rng.randint(1, min(k_atom, len(dims)) + 1)
        active_dims = rng.choice(dims, n_active, replace=False)
        W_true[i, active_dims] = rng.randn(len(active_dims))

    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)

    codes_true = np.zeros((n_samples, n_components))
    for i in range(n_samples):
        k_i = rng.randint(1, k + 1)
        idx = rng.choice(n_components, k_i, replace=False)
        codes_true[i, idx] = rng.randn(k_i)

    X = codes_true @ W_true
    scale = sigma_x / X.std()
    X *= scale
    W_true *= scale
    X += rng.randn(*X.shape) * noise_std

    return X, W_true, codes_true, dim_partition

In [ ]:
def run_single_test(dims_list, atoms_ratio, samples_ratio, sparsity_ratio_set, term3_set, kwarg_key, metrics):
    for dim in dims_list:
        for a in atoms_ratio:
            for sr in samples_ratio:
                for sparsity_ratio in sparsity_ratio_set:
                    for term3 in term3_set:                
                        key = str((dim, a, sr, sparsity_ratio, term3))
                        print("#######", key)
                        
                        atoms = math.ceil(dim*a)
                        k = atoms
                        n_samples = sr*atoms
                        active_dims = math.ceil(dim*sparsity_ratio)
                        
                        if atoms == 1:
                            print("skip: atoms=1")
                            continue
                        if key in metrics:
                            print("skip: already done, delete the shelve file if you want to do it all over again")
                            continue
            
                        device = get_device(dim)
                        X, W_true, codes_true, dim_partition = generate_synthetic_patches_sparse_atoms(
                            patch_dim=dim,
                            n_components=atoms,
                            k=k,
                            n_samples=n_samples,
                            k_atom=active_dims,
                            noise_std=0.01
                        )
                
                        scaler = B.MeanPerDimGlobalStdScaler().fit(X)
                        X_scaled = scaler.transform(X)
        
                        mets = []
                        for run_idx in range(NUM_RUNS_PER_TEST):
                            print("RUN:", run_idx)
                            kwargs = {kwarg_key: term3}
                            run = B.train(X_scaled, atoms, 1e-2, epochs=4000, device=device, **kwargs)
                            mets.append(B.get_metrics_from_run(run, W_true))
                        gc.collect()
    
                        
                        metrics[key] = B.aggregate_metrics(mets)

In [ ]:
NUM_RUNS_PER_TEST = 3

# original
DIMS_LIST = [10, 100]
ATOMS_RATIO = [0.1, 0.5, 0.9]    # for each dim, we test for 20%, 50% atoms for now, to understand how i can read the data
SAMPLES_RATIO = [1, 5, 20, 50]
SPARSITY_RATIO = [0.05, 0.2]

# test, comment for actual work
# DIMS_LIST = [10,]
# ATOMS_RATIO = [0.5, 0.9]    # for each dim, we test for 20%, 50% atoms for now, to understand how i can read the data
# SAMPLES_RATIO = [20]
# SPARSITY_RATIO = [0.2]

## Sample run

In [ ]:
X, W_true, codes_true, dim_partition = generate_synthetic_patches_sparse_atoms(
    patch_dim=25,
    n_components=5,
    k=5,
    n_samples=1000,
    k_atom=3,
    noise_std=0.01
)

scaler = B.MeanPerDimGlobalStdScaler().fit(X)
X_scaled = scaler.transform(X)

run = B.train(X_scaled, 5, 1e-2, epochs=4000)

print("first recons vs X")
S([X_scaled[0].reshape(5,5), run.recon[0].reshape(5,5)], 3, mode=MODE)
plt.show()

print("true atoms")
S([w.reshape(5,5) for w in W_true], 8, len(W_true), mode=MODE)
plt.show()

print("learned components")
S([c.reshape(5,5) for c in run.components], 8, len(W_true), mode=MODE)
plt.show()


# Relation between $\sigma_s$ and $\sigma_0$

We can either set $\sigma_s$ to $c\sigma_{\epsilon}$ and derive $\sigma_0$ from it. or the other way round.  
This will determine which one is bigger.  

The relationship we use is simply $R\sigma_0^2\sigma_s^2 = \sigma_x^2$ ($R$ is the number of atoms, or rows in components).  

- We can either set one of them relative to $sigma_eps$ and derive the other
- We can make them equal, ie. $\sigma_0 = \sigma_s = \frac{\sigma_x}{\sqrt{R}}$

Below we will empirically see that setting them to equal is a good condition, for smaller examples. The test will continue for bigger examples later.  


We test for dimensions = `[10, 100]` with `[20%, 50%, 80%]` active atoms.   

I need simple things to test on first.   


Things that change:
- number of components
- number of dims
- number of components active at a time

The simplest is checking how many atoms we can recover, we keep all the components active to keep it simple. For a given dimension.  
We will assume that data = 100 * components-number.  

In [ ]:
import math

term3_set = ["equal", "less", "greater"]
kwarg_key = "sigma_s_rel_to_0"

# dims_list, atoms_ratio, samples_ratio, term3_set, kwarg_key, metrics

with shelve.open(SHELVE_CACHE_ROOT / "t_sigma_s_rel_to_0") as shelf:
    run_single_test(DIMS_LIST, ATOMS_RATIO, SAMPLES_RATIO, SPARSITY_RATIO, term3_set, kwarg_key, shelf)
    metrics = dict(shelf)


## Results

If you look at the table, you'll see that the sigma_s_to_0 gives mean simimilary (the column `mean_sim`) = 99%.  

Surprisingly, we see that if we set $\sigma_s < \sigma_0$, `dims=10` gives 55% mean similarity. The reason is the corresponding MSE. Its much higher than the equal case, we are not able to reconstruct well, so every atom is just a disjoint noise.  

For `dims=100`, $\sigma_s > \sigma_0$, we see 30-60% similarity. The MSE is fine too. On closer inspection, you'll see that the disjoint loss has driven all atoms to near 0. If you look at individual components of a single reconstruction, you'll see random atoms lighting up. This is because the codes have too much variance and are basically taking care of getting the reconstruction.  

It's hard to say why this happens. It's best to ignore that. For now, $\sigma_s = \sigma_0$ gives the best results.  
It also has the good property to simply remove the relation with $\sigma_eps$, which we can use to freely guide reconstruction.  

The next experiments will use the "equal" method

In [ ]:
df = get_metrics_df(metrics)

In [ ]:
# all equal values
df[df["term3"] == "equal"]

In [ ]:
# all less
df[df["term3"] == "less"]

In [ ]:
# all greater
df[df["term3"] == "greater"]

# log term usefulness

Closely inspecting the gradients generally tells us that the log term has very small gradients through out.  
The log term is basically pushing in the opposite direction from disjointness. It wants to keep weights "non-zero".  
The signal is very low though compared to the actual disjoint term. And we anyways rely on correct reconstruction to give us non-zero weights.  

This section checks if descent becomes easier in the synthetic case if we give up the log term. It is useful to only check the results in the previous section with the `sigma_s_to_0 = equal` case.  


In [ ]:
term3_set = [True, False]
kwarg_key = "use_ln_term"



with shelve.open(SHELVE_CACHE_ROOT / "t_use_ln") as shelf:
    run_single_test(DIMS_LIST, ATOMS_RATIO, SAMPLES_RATIO, SPARSITY_RATIO, term3_set, kwarg_key, shelf)
    metrics = dict(shelf)


In [ ]:
df = get_metrics_df(metrics)
df

## Results

We don't see a lot of difference. Although removing the `ln` component did bring down `mean_sim` in one case (row 3).  
It has also degraded the MSE in some cases, so this is not very conclusive.  

We will need to test this on other datasets to see if there is problem with the component. Otherwise we go ahead while staying faithful to the probabilistic model.  

# Initialisations

Three kinds of initialisations:

- Standard initialisation, starts decoder and encoder with from Gaussian, mean=0 and heuristics for sigma
- SVD: initialise decoder using SVD, with sigma using the same heuristics as standard. mean=0, encoder done in the same way as standard
- Warmup: initialise using standard, then do warmup epochs on L2 regularisation (without Weight loss). Then start training

In [ ]:
# term3_set = [B.StandardInitStrategy(), B.SvdInitStrategy(), B.WarmupInitStrategy(warmup_epochs=3000)]
term3_set = [B.WarmupInitStrategy(warmup_epochs=3000)]
kwarg_key = "init_strategy"

with shelve.open(SHELVE_CACHE_ROOT / "t_inits_warmups") as shelf:
    run_single_test(DIMS_LIST, ATOMS_RATIO, SAMPLES_RATIO, SPARSITY_RATIO, term3_set, kwarg_key, shelf)
    metrics = dict(shelf)


In [ ]:
df = get_metrics_df(metrics)
df

## Results

not much difference in the pure case